# Data Preparation Day-1

Agenda Hari 1:
- Sesi 1: Why Data Preparation Matters
- Sesi 2: Acquiring Data
- Sesi 3: EDA — Kenali Datamu
- Sesi 4: Data Cleansing
- Sesi 5: Advanced Imputation
- Rekap Hari 1 + Q&A

## Why data preparation matters?


### CRISP DM Framework

1. Business Understanding
2. Data Understanding
3. Data Preparation
4. Modelling
5. Evaluation
6. Deployment

## Acquiring Data

In [ ]:
# Install required libraries
# !pip install -r requirements.txt

In [ ]:
import pandas as pd

# CSV 
df = pd.read_csv('data/data.csv')

# Dengan parameter penting:
df = pd.read_csv('data/data.csv',
    sep=',',
    encoding='utf-8',        # ganti 'latin-1' jika error
    na_values=[' ', 'N/A', '-'],
    parse_dates=['tanggal'])

print(str(df.dtypes))

# Excel
df = pd.read_excel('data/data.xlsx', sheet_name='Realisasi')

# JSON
df = pd.read_json('data/data.json')

# Parquet (modern, 5-10x lebih cepat dari CSV)
df = pd.read_parquet('data/data.parquet')

# Remote database
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

conn_str = os.getenv('CONN_STR').strip()
engine = create_engine(conn_str)
df = pd.read_sql("SELECT * FROM public.data", engine)

# API -JSON
import requests

url = "https://training.ekos.my.id/data"
response = requests.get(url)

if response.status_code == 200:
    data = response.json().get('data')
    df = pd.DataFrame(data)

df

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder

# Load dataset kotor
telco_data = "https://raw.githubusercontent.com/ekosup/data-preparation/refs/heads/main/data/telco_raw.csv"
df_dirty = pd.read_csv(telco_data)
# df_dirty.info()

# # Model dengan data kotor — minimal preprocessing
df_dirty['TotalCharges'] = pd.to_numeric(df_dirty['TotalCharges'], errors='coerce')
df_dirty = df_dirty.fillna(0)
df_dirty['Churn'] = df_dirty['Churn'].map({'Stayed': 0, 'Churned': 1})

df_dirty = df_dirty.dropna(subset=['Churn'])
df_dirty['Churn'] = df_dirty['Churn'].astype(int)

# Ambil hanya kolom numerik dulu untuk demo cepat
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
X_dirty = df_dirty[num_cols]
y = df_dirty['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X_dirty, y, test_size=0.2, random_state=42, stratify=y)

model_dirty = LogisticRegression(max_iter=1000)
model_dirty.fit(X_train, y_train)
# print('=== MODEL DATA KOTOR ===')
print(classification_report(y_test, model_dirty.predict(X_test)))

# # Kita akan revisit ini di akhir Hari 2 setelah data bersih
# # dan lihat perbedaan hasilnya!


## EDA

In [ ]:
import seaborn as sns
import missingno as msno
%matplotlib inline

# Distribusi kolom numerik
df_dirty.select_dtypes('number').hist(bins=30)

# Boxplot deteksi outlier
sns.boxplot(y=df_dirty['MonthlyCharges'])

# Missing values map
msno.matrix(df_dirty)
msno.heatmap(df_dirty)  # korelasi missing

In [ ]:
# Churn proportion (target variable)
df_dirty['Churn'].value_counts(normalize=True).round(3)
# Imbalanced!
# No: 0.735 Yes: 0.266

In [ ]:
df_dirty['TotalCharges']

## Data Cleansing

### [1] Memahami Tipe Data

Sebelum membersihkan data, kita perlu memahami tipe data setiap kolom.
Pandas menggunakan tipe berikut:

| Tipe Pandas | Keterangan | Contoh |
|-------------|------------|--------|
| `int64`     | Bilangan bulat | `tenure`, `SeniorCitizen` |
| `float64`   | Bilangan desimal | `MonthlyCharges`, `TotalCharges` |
| `object`    | Teks / string (atau campuran) | `gender`, `Contract` |
| `bool`      | True / False | — |
| `datetime64`| Tanggal & waktu | — |
| `category`  | Kategori efisien (hemat memori) | — |

In [ ]:
# Reload dataset mentah (sebelum ada perubahan apapun)
import pandas as pd

telco_data = "https://raw.githubusercontent.com/ekosup/data-preparation/refs/heads/main/data/telco_raw.csv"
df_dirty = pd.read_csv(telco_data)

# Ringkasan tipe data
print(df_dirty.dtypes)
print()
df_dirty.info()

### [2] Memperbaiki Tipe Data

Kolom `TotalCharges` seharusnya numerik, tetapi terbaca sebagai `object` karena
ada nilai berupa spasi kosong `' '` di beberapa baris.
Kita harus mengkonversi secara eksplisit menggunakan `pd.to_numeric`.

In [ ]:
# Lihat masalahnya
print('Tipe sebelum:', df_dirty['TotalCharges'].dtype)
print('Contoh nilai bermasalah:')
print(df_dirty[df_dirty['TotalCharges'].str.strip() == ''][['customerID', 'TotalCharges']].head())

# Perbaiki: paksa ke numerik, nilai yang tidak bisa dikonversi jadi NaN
df_dirty['TotalCharges'] = pd.to_numeric(df_dirty['TotalCharges'], errors='coerce')
print('\nTipe sesudah:', df_dirty['TotalCharges'].dtype)
print('Missing values di TotalCharges:', df_dirty['TotalCharges'].isna().sum())

### [3] Menghapus Data Duplikat

Data duplikat dapat muncul akibat proses ETL yang berulang atau join yang salah.
Gunakan `.duplicated()` untuk mendeteksi dan `.drop_duplicates()` untuk menghapus.

In [ ]:
# Cek duplikat
print('Jumlah baris duplikat:', df_dirty.duplicated().sum())
print('Jumlah baris duplikat (berdasarkan customerID):', df_dirty.duplicated(subset='customerID').sum())

# Hapus duplikat — keep='first' mempertahankan kemunculan pertama
df_dirty = df_dirty.drop_duplicates(subset='customerID', keep='first')
print('Jumlah baris setelah drop_duplicates:', len(df_dirty))

### [4] Menangani Missing Values

Ada dua strategi utama:

| Strategi | Kapan digunakan | Fungsi |
|----------|----------------|--------|
| **Drop** | Data hilang < 5%, kolom tidak penting | `.dropna()` |
| **Impute** | Data hilang > 5%, kolom penting | `.fillna()` / SimpleImputer |

In [ ]:
# Ringkasan missing values
missing = df_dirty.isnull().sum()
missing_pct = (missing / len(df_dirty) * 100).round(2)
print(missing[missing > 0].to_frame('count').assign(pct=missing_pct))

# Strategi 1 — Drop baris yang TotalCharges-nya NaN (hanya ~11 baris, < 1%)
df_drop = df_dirty.dropna(subset=['TotalCharges'])
print('\nSetelah dropna:', len(df_drop), 'baris')

# Strategi 2 — Imputasi numerik dengan median
median_tc = df_dirty['TotalCharges'].median()
df_imputed = df_dirty.copy()
df_imputed['TotalCharges'] = df_imputed['TotalCharges'].fillna(median_tc)
print('Imputasi median TotalCharges:', median_tc)

### [5] Pipeline Lengkap — `clean_df`

Setelah memahami setiap langkah secara terpisah, kita gabungkan semuanya
ke dalam satu fungsi yang bisa digunakan ulang.

In [ ]:
import pandas as pd

def clean_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 1. Fix tipe data
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

    # 2. Drop NA hanya jika masih ada
    df = df.dropna(subset=['TotalCharges'])

    # 3. Impute numerik (median)
    if 'MonthlyCharges' in df.columns:
        median_val = df['MonthlyCharges'].median()
        df['MonthlyCharges'] = df['MonthlyCharges'].fillna(median_val)

    # 4. Impute kategorikal (modus)
    if 'MultipleLines' in df.columns:
        if not df['MultipleLines'].mode().empty:
            mode_val = df['MultipleLines'].mode()[0]
            df['MultipleLines'] = df['MultipleLines'].fillna(mode_val)

    # 5. Final check
    if df.isnull().sum().sum() != 0:
        raise ValueError("Masih ada missing values!")

    return df

In [ ]:
df = clean_df(df_dirty)

df.head()

## Penanganan Outlier

**Outlier** adalah nilai yang jauh menyimpang dari distribusi normal data.
Keberadaannya bisa disebabkan oleh:
- Kesalahan input / pengukuran
- Kejadian ekstrem yang valid (misal pelanggan dengan tagihan sangat tinggi)

Sebelum menghapus atau mengubah outlier, selalu tanyakan: **apakah nilai ini masuk akal secara bisnis?**

### [1] Mendeteksi Outlier

Ada dua metode deteksi yang umum digunakan:

| Metode | Cara Kerja | Cocok untuk |
|--------|-----------|-------------|
| **IQR** (Interquartile Range) | Outlier = nilai di luar `[Q1 − 1.5×IQR, Q3 + 1.5×IQR]` | Distribusi skewed / tidak normal |
| **Z-Score** | Outlier = nilai dengan `\|z\| > 3` (> 3 stddev dari mean) | Distribusi mendekati normal |

Visualisasi paling cepat untuk melihat outlier adalah **boxplot**.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ['tenure', 'MonthlyCharges', 'TotalCharges']):
    sns.boxplot(y=df[col], ax=ax)
    ax.set_title(col)
plt.suptitle('Distribusi kolom numerik — outlier terlihat sebagai titik di luar whisker')
plt.tight_layout()
plt.show()

### [2] Menangani Outlier

Setelah terdeteksi, ada tiga pilihan penanganan:

| Strategi | Kapan digunakan | Cara |
|----------|----------------|------|
| **Drop** | Outlier akibat error data, jumlah sedikit | `df[mask]` — filter baris di luar batas |
| **Winsorizing / Capping** | Outlier valid tapi ekstrem, ingin mempertahankan baris | `.clip(lower, upper)` |
| **Transformasi** | Distribusi sangat skewed | `np.log1p()`, `np.sqrt()` |

In [ ]:
# Metode 1: IQR
Q1 = df['MonthlyCharges'].quantile(0.25)
Q3 = df['MonthlyCharges'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

# Metode 2: Z-Score
from scipy import stats
z = abs(stats.zscore(df['tenure']))
outliers = df[z > 3]

# Winsorizing (capping) — ganti dengan batas IQR
df['charges_capped'] = df['MonthlyCharges'].clip(lower=lower, upper=upper)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Plot pertama
sns.boxplot(y=df['MonthlyCharges'], ax=axes[0])
axes[1].set_title('Monthly Charges - Before Winsorizing')

# Plot kedua
sns.boxplot(y=df['charges_capped'], ax=axes[1])
axes[0].set_title('Charges Capped - After Winsorizing')

plt.tight_layout()
plt.show()

---
## Real-Time Data Pipeline

Pada sesi ini kita akan mensimulasikan pipeline data **sensor IoT** yang masuk secara
real-time, mulai dari akuisisi hingga dashboard monitoring — mengikuti alur **CRISP-DM**.

```
API Endpoint  →  [1] Business Understanding
              →  [2] Data Understanding  (fetch & explore)
              →  [3] Data Preparation   (cleanse, fix, outlier)
              →  [4] Dashboard          (visualisasi hasil bersih)
```

> **Simulasi stream:** setiap kali kamu memanggil endpoint, kamu mendapat 10 baris baru.
> Jalankan cell akuisisi berulang kali untuk mensimulasikan data yang terus masuk.

### [1] Business Understanding

| | |
|---|---|
| **Konteks** | Jaringan sensor IoT dipasang di 5 kota besar Indonesia untuk memantau kondisi lingkungan secara real-time. |
| **Problem** | Data mentah dari sensor sering kotor: nilai null, tipe data salah, duplikat, dan pembacaan ekstrem (outlier). |
| **Tujuan** | Membangun pipeline otomatis yang membersihkan data begitu masuk, lalu menampilkan dashboard kondisi terkini. |
| **Kriteria sukses** | Dashboard terupdate setiap batch, tidak ada nilai null atau tipe data salah di output akhir. |

**Skema data sensor & imperfeksi yang diketahui:**

| Kolom | Tipe Harapan | Imperfeksi | Frekuensi |
|-------|-------------|------------|-----------|
| `sensor_id` | `str` | Format rusak, misal `SEN-3_ERROR` | ~12% |
| `location` | `str` | — | — |
| `temperature_c` | `float` | `null` + outlier fisik (80–150 °C) | ~15% null, ~10% outlier |
| `humidity_pct` | `float` | String `"N/A"` + `null` | ~10% dirty, ~5% null |
| `pressure_hpa` | `float` | `null` + outlier fisik (1100–1300 hPa) | ~10% null, ~8% outlier |
| `battery_pct` | `float` | `null` | ~5% |
| `signal_strength` | `str` | `"UNKNOWN"` atau `null`; nilai valid: `strong`, `medium`, `weak` | bervariasi |
| `reading_count` | `int` | String `"error"` (mixed type) | ~8% |

### [2] Data Understanding — Fetch & Eksplorasi

Langkah pertama: ambil data dari endpoint dan pahami kondisi aktualnya.

In [ ]:
import requests
import pandas as pd

ENDPOINT = "https://training.ekos.my.id/api/realtime-data"

def fetch_batch() -> pd.DataFrame:
    """Ambil satu batch (10 baris) dari endpoint sensor."""
    resp = requests.get(ENDPOINT, timeout=10)
    resp.raise_for_status()
    payload = resp.json()
    return pd.DataFrame(payload['data'])

# Ambil satu batch untuk eksplorasi
df_raw = fetch_batch()
print(f"Jumlah baris: {len(df_raw)}")
df_raw

In [ ]:
# Tipe data awal — sebelum apapun
print("=== dtypes ===")

print()

# Missing values (termasuk string 'N/A', 'error', 'UNKNOWN')
print("=== Nilai null aktual ===")

print()

# Nilai unik per kolom kategorikal — tampak imperfeksinya
print("=== signal_strength — nilai unik ===")

print()
print("=== humidity_pct — nilai unik (sample) ===")

print()
print("=== reading_count — nilai unik ===")

print()
print("=== sensor_id — semua nilai ===")

print()

# Deteksi sensor_id dengan format rusak (valid: SEN-XXX)
print()

# Deteksi outlier temperature_c (tidak mungkin 80-150°C di Indonesia)
print()

# Deteksi outlier pressure_hpa

### [3] Data Preparation — Pipeline Pembersihan

Dari eksplorasi di atas kita menemukan **7 masalah** yang harus ditangani secara berurutan:

| # | Masalah | Kolom | Strategi |
|---|---------|-------|----------|
| 1 | `sensor_id` format rusak (`SEN-3_ERROR`) | `sensor_id` | Filter pakai regex, baris invalid → drop |
| 2 | String tidak valid (`"N/A"`, `"error"`) | `humidity_pct`, `reading_count` | Cast ke numerik → NaN |
| 3 | Standarisasi kategori (`"UNKNOWN"`, `null`) | `signal_strength` | Lowercase + peta ke nilai valid atau `"unknown"` |
| 4 | Outlier fisik `temperature_c` (80–150 °C tidak mungkin) | `temperature_c` | Nilai di luar range → NaN sebelum imputasi |
| 5 | Outlier fisik `pressure_hpa` (> 1100 hPa tidak realistis) | `pressure_hpa` | Winsorizing ke batas fisik valid |
| 6 | Missing values numerik (~5–15%) | semua kolom float | Imputasi median per lokasi, fallback global |
| 7 | Tipe data akhir tidak sesuai | `reading_count` | Cast ke `Int64` (nullable int) |

In [ ]:
import numpy as np
from scipy import stats

NUMERIC_COLS = ['temperature_c', 'humidity_pct', 'pressure_hpa', 'battery_pct', 'reading_count']
PRESSURE_MAX = 1100.0   # hPa — batas fisik realistis di permukaan bumi
PRESSURE_MIN = 870.0

def clean_sensor_batch(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # --- Langkah 1: Cast ke numerik (string 'N/A', 'error' → NaN) ---

    # --- Langkah 2: Standarisasi signal_strength ---

    # --- Langkah 3: Imputasi missing values dengan median per lokasi ---

    # reading_count: imputasi dengan median global

    # --- Langkah 4: Outlier pressure_hpa — cap ke batas fisik realistis ---

    # --- Langkah 5: Hapus duplikat sensor_id — keep last (paling baru) ---

    # --- Langkah 6: Perbaiki tipe akhir ---

    # --- Validasi akhir ---

    return df


# Jalankan pipeline pada batch yang sudah kita fetch
df_clean = clean_sensor_batch(df_raw)

print(f"Raw: {len(df_raw)} baris → Clean: {len(df_clean)} baris")
print(f"Null tersisa: {df_clean.isnull().sum().sum()}")
df_clean

In [ ]:
# Verifikasi tipe data akhir
print(df_clean.dtypes)
print()
df_clean.describe()

### [4] Dashboard — Monitoring Real-Time

Setelah data bersih, kita buat dashboard yang **polling otomatis setiap 5 detik**.
Tekan tombol **⏹ Stop** untuk menghentikan loop, atau interrupt kernel (`■`) kapan saja.

> **Cara kerja:** setiap siklus memanggil `fetch_batch()` → `clean_sensor_batch()` → render ulang dashboard.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

def run_dashboard(n_batches: int = 1):
    """
    Fetch n_batches, jalankan pipeline, tampilkan dashboard.
    Catatan: clear_output ditangani di luar fungsi agar widget tidak ikut terhapus.
    """
    frames = []
    for _ in range(n_batches):
        raw = fetch_batch()
        clean = clean_sensor_batch(raw)
        frames.append(clean)

    data = pd.concat(frames, ignore_index=True)

    fig = plt.figure(figsize=(16, 10))
    fig.suptitle(f'Sensor Dashboard  |  {len(data)} sensor readings  |  {n_batches} batch(es)',
                 fontsize=14, fontweight='bold')
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

    # --- Plot 1: Suhu rata-rata per kota ---

    # --- Plot 2: Kelembaban rata-rata per kota ---

    # --- Plot 3: Distribusi signal strength ---

    # --- Plot 4: Battery level per sensor ---

    # --- Plot 5: Pressure distribution (boxplot per kota) ---

    # --- Plot 6: Tabel ringkasan KPI ---

    plt.show()
    plt.close(fig)  # bebaskan memori


In [ ]:
run_dashboard(n_batches=1)

In [ ]:
# ── Real-time polling: update dashboard setiap 5 detik ─────────────────────
# Tombol Stop bekerja karena dashboard berjalan di background thread,
# bukan di main kernel thread — sehingga widget event tetap diproses.
import time, threading
import ipywidgets as widgets  # https://pypi.org/project/ipywidgets/
from IPython.display import display

